In [1]:
!pwd

/Users/anshulchiranth/Desktop/Strike/Voice Experiments


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("AAI_API_KEY")

In [3]:
import assemblyai as aai
aai.settings.api_key = api_key

In [4]:
from pathlib import Path
import requests
import time

base_dir = Path.cwd()

transaction_dir = base_dir / "Unclipped Processed Transactions"

In [5]:
files = [f for f in transaction_dir.iterdir()
               if f.is_file() and not f.name.startswith(".")]

In [6]:
base_url = "https://api.assemblyai.com"

headers = {
"authorization": api_key
}

In [7]:
#Collect Transcripts (no timestamps) for each file in files
#Enforce 2 Speakers

transcripts = {}

for file in files:
    with open(file, "rb") as f:
        response = requests.post(base_url + "/v2/upload", headers=headers, data=f)
        if response.status_code != 200:
            print(f"Error: {response.status_code}, Response: {response.text}")
            response.raise_for_status()
        upload_json = response.json()
        audio_file = upload_json["upload_url"]


    data = {
    "audio_url": audio_file, # You can also use a URL to an audio or video file on the web
    "speech_models": ["universal-3-pro", "universal-2"],
    "language_detection": True,
    "speaker_labels": True,
    "speakers_expected": 2
    }

    response = requests.post(base_url + "/v2/transcript", headers=headers, json=data)
    transcript_id = response.json()["id"]
    polling_endpoint = base_url + f"/v2/transcript/{transcript_id}"

    while True:
        transcript = requests.get(polling_endpoint, headers=headers).json()
        if transcript["status"] == "completed":
            break
        elif transcript["status"] == "error":
            raise RuntimeError(f"Transcription failed: {transcript['error']}")
        else:
            time.sleep(3)

    transcripts[file] = transcript

In [8]:
#Test to see if just one utterance
one_utterance = []

for file, transcript in transcripts.items():
    if len(transcript["utterances"]) == 1:
        one_utterance.append(file)

In [9]:
#Test to see if more than two speakers
two_plus_speakers = []

for file, transcript in transcripts.items():
    speaker_set = set()
    for utterance in transcript["utterances"]:
        speaker = utterance["speaker"]
        if speaker not in speaker_set:
            speaker_set.add(speaker)

    if len(speaker_set) > 2:
        two_plus_speakers.append(file)

In [13]:
one_utterance

[PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/julian_tx_4.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/jake_tx_2.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/jake_tx_1.wav'),
 PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/George_tx_5.wav')]

In [23]:
for file_name in one_utterance:
    t = transcripts[file_name]
    print(file_name, t["id"])

/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/julian_tx_4.wav 48337230-b634-4927-9456-e2ee4b83852c
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/jake_tx_2.wav dee5b988-8450-43a5-98c2-a358c9ffebfb
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/jake_tx_1.wav c4a88a80-14ae-40f0-9777-b40d94330e7e
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/George_tx_5.wav 5edbe177-bbdf-4143-9f4a-7614478a9d7f


In [26]:
for file_name in two_plus_speakers:
    t = transcripts[file_name]
    print(file_name, t["id"])
    print(type(t["id"]))

/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/jake_tx_9.wav 95adbe4d-0e59-42c8-aba2-d8b64844e081
<class 'str'>


In [14]:
two_plus_speakers

[PosixPath('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/jake_tx_9.wav')]

In [19]:
t = transcripts[Path('/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/jake_tx_9.wav')]

for utterance in t["utterances"]:
    print(f"{utterance['speaker']}: {utterance['text']}")

B: All right, welcome to Dairy Queen. What can I get for you tonight?
A: Hey, um, can I please get a mini— no, I'll do a small chocolate blizzard with cookie dough and Oreo cookies, please.
C: Anything else for you tonight?
A: That's it.
C: It's going to be $6.37 at the window.
A: Thank you.
C: You're welcome.


In [20]:
t["id"]

'95adbe4d-0e59-42c8-aba2-d8b64844e081'

In [21]:
for file_name, transcript in transcripts.items():
    print(file_name, transcript["id"])

/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/julian_tx_4.wav 48337230-b634-4927-9456-e2ee4b83852c
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/julian_tx_5.wav 02e37fe7-c890-474b-8de8-9efe2a685bbd
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/julian_tx_7.wav 16c07be3-ff97-49d1-904c-c8792962d0ed
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/julian_tx_6.wav 739b5f72-5f13-4a1f-a20e-180337756104
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/julian_tx_2.wav 64247769-b08d-477b-a1ab-d80d99a7de4a
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/julian_tx_3.wav 83a2c08f-4717-4505-b8b5-d06b081668c5
/Users/anshulchiranth/Desktop/Strike/Voice Experiments/Unclipped Processed Transactions/julian_tx_1.wav 7c14ceaa-5d97-4c32-ae43-09812d686dee
/Users/anshul